# Figure 6 – Error propagation

Data are read from `data/` (written by `run_all.py`). Every figure is saved as `Figure6_<nn>_<title>.png` when it is shown.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

DATA = Path("data")
PREFIX = "Figure6"
_plt_show = plt.show
_n = [0]


def _title(fig):
    for ax in fig.axes:
        t = ax.get_title()
        if t:
            return re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_")[:50]
    return "figure"


def _save_then_show(*args, **kwargs):
    for num in plt.get_fignums():
        fig = plt.figure(num)
        _n[0] += 1
        name = f"{PREFIX}_{_n[0]:02d}_{_title(fig)}.png"
        fig.savefig(name, dpi=fig.dpi, bbox_inches="tight")
        print("saved:", name)
    _plt_show(*args, **kwargs)


plt.show = _save_then_show

In [ ]:
res_impact = pd.read_csv(DATA / "res_impact.csv")
df_soc_err = pd.read_csv(DATA / "df_soc_err.csv")
df_soh_err = pd.read_csv(DATA / "df_soh_err.csv")
df_gain = pd.read_csv(DATA / "df_gain.csv")

## Absolute SOH error in three scenarios (box plot, first version)

Control: predicted material and SOC; SOC bias: true material with SOC × 1.3; wrong material: the first other material in the order NMC, LMO, LFP.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

plt.figure(figsize=(5, 4), dpi=600)

plot_df = res_impact.melt(value_vars=["Normal_SOH_Error", "SOH_Error_with_SOC_Bias", "SOH_Error_on_Wrong_Mat"],
                          var_name="Scenario", value_name="SOH_MAE")

sns.boxplot(x="Scenario", y="SOH_MAE", data=plot_df, palette="muted")

plt.xticks([0, 1, 2], ["Control\n(Best Case)", "SOC Error Impact\n(Mat Correct)", "Material Error Impact\n(Cascading Error)"])
plt.ylabel("Absolute Error in SOH")
plt.show()

## SOC error when the material is wrong

In [ ]:
plt.figure(figsize=(3,2),dpi=600)
for key, g in df_soc_err.groupby(["true_mat", "assumed_mat"]):
    label = f"{key[0]} → {key[1]}"
    plt.hist(g["SOC_error"], bins=40, alpha=0.5, label=label)

plt.xlabel("SOC Error")
plt.ylabel("Count")
plt.legend(fontsize=6, frameon=False, loc="upper left", bbox_to_anchor=(1.02, 1.0))
plt.tight_layout()
plt.show()

## SOC error vs. SOH error when the material is wrong

In [ ]:
plt.figure(figsize=(4,3),dpi=600)

for key, g in df_soh_err.groupby(["true_mat", "assumed_mat"]):
    plt.scatter(
        g["SOC_error"], g["SOH_error"],
        alpha=0.4, label=f"{key[0]} → {key[1]}"
    )

plt.xlabel("SOC Error")
plt.ylabel("SOH Error")
plt.tight_layout()
plt.show()

## SOH error under SOC perturbation (true material) and summary of the wrong-material cases

In [ ]:
plt.figure(figsize=(6, 5),dpi=600)

color_map = {
    "NMC": "#86BAD4",
    "LMO": "#F5B783",
    "LFP": "#E0795F",
}

for mat, g in df_gain.groupby("Mat"):
    mean_curve = g.groupby("SOC_delta")["SOH_delta"].mean()
    plt.plot(
        mean_curve.index,
        mean_curve.values,
        marker="o",
        linewidth=2,
        markersize=5,
        color=color_map.get(mat, "gray"),
        label=mat
    )

plt.xlabel("Injected SOC Error")
plt.ylabel("Resulting SOH Error")
plt.title("Material-dependent SOC → SOH Error Gain")
plt.tight_layout()
plt.show()

summary = []

for key, g in df_soh_err.groupby(["true_mat", "assumed_mat"]):
    summary.append({
        "Case": f"{key[0]}→{key[1]}",
        "Mean_SOC_Error": g["SOC_error"].mean(),
        "Std_SOC_Error": g["SOC_error"].std(),
        "Mean_SOH_Error": g["SOH_error"].mean(),
        "Std_SOH_Error": g["SOH_error"].std()
    })

df_summary = pd.DataFrame(summary)
print(df_summary)

## Absolute SOH error in three scenarios (box plot, final version)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(3, 2), dpi=600)

plot_df = res_impact.melt(
    value_vars=[
        "Normal_SOH_Error",
        "SOH_Error_with_SOC_Bias",
        "SOH_Error_on_Wrong_Mat"
    ],
    var_name="Scenario",
    value_name="SOH_MAE"
)

order = [
    "Normal_SOH_Error",
    "SOH_Error_with_SOC_Bias",
    "SOH_Error_on_Wrong_Mat"
]

palette = [
    "#BDBDBD",
    "#86BAD4",
    "#E0795F",
]

flierprops = dict(
    marker='o',
    markerfacecolor='0.5',
    markeredgecolor='0.5',
    markersize=2,
    linestyle='none',
    alpha=0.6
)

sns.boxplot(
    x="Scenario",
    y="SOH_MAE",
    data=plot_df,
    order=order,
    palette=palette,
    width=0.6,
    showfliers=True,
    flierprops=flierprops
)

plt.xticks(
    [0, 1, 2],
    [
        "(Best Case)",
        "(Mat Correct)",
        "(Cascading)"
    ]
)

plt.ylabel("Absolute SOH Error")
plt.xlabel("")
plt.tight_layout()
plt.show()